# Combining the Spectra of the 3 EPIC Cameras -- Part 1: Filtering the Observation
<hr style="border: 2px solid #f5bf03" />

- **Description:** Step-by-step guide to combine the spectra of all three EPIC camera exposures into one single spectrum with corresponding rmf, arf and bkg files.
- **Level:** Intermediate
- **Data:** XMM observation of the Circinus Galaxy (obsid=0111240101)
- **Requirements:** Must be run using pySAS version 2.2.7 or higher.
- **Credit:** Ryan Tanner (October 2025), based on an <a href="https://www.cosmos.esa.int/web/xmm-newton/sas-thread-epic-merging">ESA SOC SAS Tutorial</a>
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/cgi-bin/Feedback">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 30 June 2026, for SAS v22.1 and pySAS v2.5.0

<hr style="border: 2px solid #f5bf03" />

## 1. Introduction

This example is divided into two parts. Part 1 deals with filtering the observation and preparing the spectra. Part 2 deals with combining the spectra and analyzing it with XSPEC.

A lot of the explanation about the filtering and SAS tasks used will be omitted as they are covered in other notebooks (e.g. *ABC Guide for XMM-Newton -- EPIC Image Creation and Basic Filtering* and *ABC Guide for XMM-Newton -- EPIC Source Extraction and Spectrum Creation*).

#### SAS Tasks to be Used

- `emproc`[(Documentation for emproc)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/emproc/index.html "emproc Documentation")
- `epproc`[(Documentation for epproc)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/epproc/index.html "epproc Documentation")
- `evselect`[(Documentation for evselect)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/evselect/index.html)
- `espfilt`[(Documentation for espfilt)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/espfilt/index.html)
- `epatplot`[(Documentation for epatplot)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/epatplot/index.html)
- `rmfgen`[(Documentation for rmfgen)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/rmfgen/index.html)
- `arfgen`[(Documentation for arfgen)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/arfgen/index.html)
- `specgroup`[(Documentation for specgroup)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/specgroup/index.html)

#### Useful Links

- [`pysas` Documentation](https://github.com/XMMGOF/pysas_docs "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads/ "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/cgi-bin/Feedback "Helpdesk") - Link to form to contact the XMM-Newton GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

## 2. Setup

In [ ]:
# pySAS imports
import pysas
from pysas import MyTask

# Useful imports
import os, glob, shutil

# Imports for plotting
import matplotlib.pyplot as plt
from astropy.visualization import astropy_mpl_style
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from regions import CircleSkyRegion
from astropy.coordinates import SkyCoord
from astropy.visualization import ZScaleInterval, ImageNormalize
import astropy.units as u
from matplotlib.ticker import StrMethodFormatter
from IPython.display import Image, display
plt.style.use(astropy_mpl_style)

# To handle certain warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
obsid = '0111240101'
my_obs = pysas.ObsID(obsid)
my_obs.basic_setup(overwrite=False,run_rgsproc=False)

In [ ]:
event_lists = []
event_lists.append(my_obs.files['M1evt_list'][0])
event_lists.append(my_obs.files['M2evt_list'][0])
event_lists.append(my_obs.files['PNevt_list'][0])

***
The cell below will make a series of dictionaries containing filenames that will be used throughout this notebook.

In [ ]:
time_filtered_evts  = {}
clean_event_lists   = {}
hi_res_images       = {}
source_event_list   = {}
bkg_event_list      = {}
source_spectra_file = {}
bkg_spectra_file    = {}
rmf_file            = {}
arf_file            = {}
grouped_spectra     = {}
epatplot            = {}

for event_list in event_lists:
    with fits.open(event_list) as hdu:
        instrument = hdu[0].header['INSTRUME']
    time_filtered_evts[instrument]  = f'{instrument}_time_filtered_evts.fits'
    clean_event_lists[instrument]   = f'{instrument}_clean_event_list.fits'
    hi_res_images[instrument]       = f'{instrument}_image.fits'
    source_event_list[instrument]   = f'{instrument}_source_event_list.fits'
    bkg_event_list[instrument]      = f'{instrument}_bkg_event_list.fits'
    source_spectra_file[instrument] = f'{instrument}_pi.fits'
    bkg_spectra_file[instrument]    = f'{instrument}_bkg_pi.fits'
    rmf_file[instrument]            = f'{instrument}_rmf.fits'
    arf_file[instrument]            = f'{instrument}_arf.fits'
    grouped_spectra[instrument]     = f'{instrument}_grp.fits'
    epatplot[instrument]            = f'{instrument}_epat.pdf'

filepha="src_spectrum_grp.ds"
filebkg="bkg_spectrum_grp.ds"
filersp="response_grp.rmf"

***
The cell below contains a number of functions that will be used throughout this notebook.

In [ ]:
def filter_event_list(in_event_list,
                      pi_min,
                      pi_max,
                      filtered_event_list,
                      pattern = None):

    with fits.open(in_event_list) as hdu:
        instrument = hdu[0].header['INSTRUME']

    if instrument == 'EPN':
        filter = 'XMMEA_EP'
        if pattern is None: pattern = 4
    elif 'EMOS' in instrument:
        filter = 'XMMEA_EM'
        if pattern is None: pattern = 12

    # Filter expression
    expression = '(PATTERN in [0:{pattern}])&&(PI in [{pi_min}:{pi_max}])&&(FLAG == 0)&&#{filter}'.format(filter=filter,pattern=pattern,pi_min=pi_min,pi_max=pi_max)

    inargs = {'table'           : in_event_list, 
              'withfilteredset' : 'yes', 
              'expression'      : expression, 
              'filteredset'     : filtered_event_list, 
              'filtertype'      : 'expression', 
              'keepfilteroutput': 'yes', 
              'updateexposure'  : 'yes', 
              'filterexposure'  : 'yes'}
    
    MyTask('evselect', inargs).run()

def make_hires_image(in_event_list,
                     pi_min = 200,
                     pi_max = 13000,
                     out_image='image.fits',
                     output   = True):

    # Filter expression
    expression = '(PI in [{pi_min}:{pi_max}])'.format(pi_min=pi_min,pi_max=pi_max)

    inargs = {'table'         : in_event_list+':EVENTS', 
              'withimageset'  : 'yes',
              'expression'    : expression, 
              'imageset'      : out_image,
              'imagebinning'  : 'binSize',
              'xcolumn'       : 'X',
              'ycolumn'       : 'Y',
              'ximagebinsize' : 40,
              'yimagebinsize' : 40}
    
    MyTask('evselect', inargs, output_to_terminal = output).run()

    return out_image

def plot_region(image_file, ra, dec, radius, vmin=1.0, vmax=1000.0, zoom = 20.0):
    
    # Define region
    center = SkyCoord(ra, dec)
    region = CircleSkyRegion(center, radius)
    
    # Open file
    hdu = fits.open(image_file)[0]
    wcs = WCS(hdu.header)
    instrument = hdu.header['INSTRUME']

    # Convert region to artist object
    pixel_region = region.to_pixel(wcs)
    artist = pixel_region.as_artist(color='lime')

    # Set image limits
    # This sets the bounds of the lower left (ll) and upper right (ur) of the plot.
    # NOTE: The calculation for the ll and ur of RA is reversed from the
    # calculation for the ll and ur of the DEC (+,- vs. -,+).
    # This preserves the correct orientation of the image.
    # The limits for the RA are also double the limits for the DEC to preserve
    # the aspect ratio.
    ra_ll  = ra+2*zoom*radius
    ra_ur  = ra-2*zoom*radius
    dec_ll = dec-zoom*radius
    dec_ur = dec+zoom*radius
    ra_lim  = [ra_ll.value, ra_ur.value]
    dec_lim = [dec_ll.value, dec_ur.value]
    # The third value "0" sets the "origin", or the index of the first pixel value.
    # It is "0" because Python starts counting at "0".
    (xmin, xmax), (ymin, ymax) = wcs.all_world2pix(ra_lim, dec_lim, 0)

    # Plot
    ax = plt.subplot(projection=wcs)
    plt.imshow(hdu.data, origin='lower', norm='log', vmin=vmin, vmax=vmax)
    ax.set_facecolor("black")
    ax.add_artist(artist)
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
    plt.grid(color='blue', ls='solid')
    plt.xlabel('RA')
    plt.ylabel('Dec')
    plt.title(f'{instrument} Image with Region')
    plt.colorbar()
    plt.show()

def plot_multi_regions(image_file, sources, ra_cent, dec_cent, radius_cent, vmin=1.0, vmax=1000.0, zoom = 7.0):
    # Here 'sources' is a list of tuples containing the RA, Dec, and radius of each source region. 
    # The image will be centered on (ra_cent, dec_cent), with zoom set by radius_cent and zoom.

    # Open file
    hdu = fits.open(image_file)[0]
    wcs = WCS(hdu.header)
    instrument = hdu.header['INSTRUME']

    # Set image limits
    # This sets the bounds of the lower left (ll) and upper right (ur) of the plot.
    # NOTE: The calculation for the ll and ur of RA is reversed from the
    # calculation for the ll and ur of the DEC (+,- vs. -,+).
    # This preserves the correct orientation of the image.
    # The limits for the RA are also double the limits for the DEC to preserve
    # the aspect ratio.
    ra_ll  = ra_cent+2*zoom*radius_cent
    ra_ur  = ra_cent-2*zoom*radius_cent
    dec_ll = dec_cent-zoom*radius_cent
    dec_ur = dec_cent+zoom*radius_cent
    ra_lim  = [ra_ll.value, ra_ur.value]
    dec_lim = [dec_ll.value, dec_ur.value]
    # The third value "0" sets the "origin", or the index of the first pixel value.
    # It is "0" because Python starts counting at "0".
    (xmin, xmax), (ymin, ymax) = wcs.all_world2pix(ra_lim, dec_lim, 0)

    # Create plot
    ax = plt.subplot(projection=wcs)
    plt.imshow(hdu.data, origin='lower', norm='log', vmin=vmin, vmax=vmax)
    ax.set_facecolor("black")
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))

    # Add regions
    artist = []
    for i, source in enumerate(sources):
        # Define region
        center = SkyCoord(source[0], source[1])
        region = CircleSkyRegion(center, source[2])
        
        # Convert region to artist object
        pixel_region = region.to_pixel(wcs)
        artist.append(pixel_region.as_artist(color='lime'))

        # Add region as artist
        ax.add_artist(artist[i])

    # Finish Plot
    plt.grid(color='blue', ls='solid')
    plt.xlabel('RA')
    plt.ylabel('Dec')
    plt.title(f'{instrument} Image with Regions')
    plt.colorbar()
    plt.show()

def plot_zoom_in(image_file, zoom=4, x=None, y=None, vmin=1.0, vmax=10.0):
    # Open file
    hdu = fits.open(image_file)[0]
    wcs = WCS(hdu.header)
    instrument = hdu.header['INSTRUME']
    im_shape = hdu.shape
    if x is None:
        x_center = int(im_shape[0]/2)
    else:
        x_center = x
    if y is None:
        y_center = int(im_shape[1]/2)
    else:
        y_center = y
    
    # Define the zoomed-in region
    xmin, xmax = x_center-int(x_center/(2*zoom)), x_center+int(x_center/(2*zoom))
    ymin, ymax = y_center-int(y_center/(2*zoom)), y_center+int(y_center/(2*zoom))

    # Plot
    ax = plt.subplot(projection=wcs)
    plt.imshow(hdu.data, origin='lower', norm='log', vmin=vmin, vmax=vmax)
    ax.set_facecolor("black")
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
    plt.grid(color='blue', ls='solid')
    plt.xlabel('RA')
    plt.ylabel('Dec')
    plt.title(f'{instrument} Image')
    plt.colorbar()
    plt.show()

## 3. Begin Filtering

Check the status of the unfiltered event lists.

In [ ]:
for file in event_lists:
    my_obs.quick_eplot(file)

Use `espfilt` to filter for particle flaring.

In [ ]:
for file in event_lists:
    with fits.open(file) as hdu:
        inst = hdu[0].header['INSTRUME']
    if inst == 'EPN':
        rangescale = 15
    elif 'EMOS' in inst:
        rangescale = 6
    inargs = {'eventfile'  : file,
              'rangescale' : rangescale,
              'elow'       : 500,
              'ehigh'      : 11995}
    MyTask('espfilt', inargs).run()

Files with the name `*allevc.fits` contain the time filtered event lists. (We will make copies of the files and give them filenames with the form `{instrument}_time_filtered_evts.fits`.) We will use these to then apply other filters. For now we will use `PATTERN<=12` for the MOS and `PATTERN<=4` for the pn right now. Later on we will change this to `PATTERN=0` for both the MOS and pn. 

In [ ]:
cevent_lists = glob.glob('*allevc.fits')
attitude_file = glob.glob('*AttHk.ds')[0]

for event_list in cevent_lists:
    with fits.open(event_list) as hdu:
        inst = hdu[0].header['INSTRUME']
    shutil.copy(event_list, time_filtered_evts[inst])

In [ ]:
cevent_lists

In [ ]:
gti_list = glob.glob('*-gti.fits')

In [ ]:
gti_list

In [ ]:
pimin = 200
pimax = 13000

for inst, file in time_filtered_evts.items():
    filter_event_list(file,pimin,pimax,clean_event_lists[inst])

Let's check the filtered event lists. The pn shows out-of-time contamination (the stripe going down from central source). Because we are interested in the main source we are not concered about the out-of-time contamination since it doesn't affect us. If out-of-time filtering is needed you can check out the tutorials on dealing with out-of-time events.

In [ ]:
for inst, file in clean_event_lists.items():
    my_obs.quick_eplot(file)

Let's take a closer look at the central source.

In [ ]:
# Make high resolution images from the three clean event lists for the pn, MOS1, and MOS2
for inst, file in clean_event_lists.items():
    make_hires_image(file,pimin,pimax,out_image=hi_res_images[inst])

In [ ]:
for inst, file in hi_res_images.items():
    plot_zoom_in(file)

That looks horrible. Let's adjust the z scale.

In [ ]:
for inst, file in hi_res_images.items():
    plot_zoom_in(file, vmax = 1000.0)

What initially appeared to be one bright source is actually three (or more!) distinct sources! Let's select the brightest source.

<div class="alert alert-block alert-info">
    <b>Note:</b> There are two ways of selecting region for the brightest source. We can use a source radius of 11 arcseconds which would cover the source and most of the psf, but an alternative would be to use a source radius of 30 arcseconds. This would encompass the two nearest sources which you would have to remove by defining two smaller regions with radii of 7 arcseconds.
</div>

A little bit further down we will do the extraction of the source region using `evselect`. To select just the central source use the following expression:

```
expression = "'((RA,DEC) in CIRCLE(source_RA,source_Dec,source_rad))'"
```

To use a larger source radius and remove the two smaller sources the expression would look something like this:

```
expression = "'((RA,DEC) in CIRCLE(source_RA,source_Dec,source_rad) &&! ((RA,DEC) in CIRCLE(source2_RA,source2_Dec,source2_rad)) &&! ((RA,DEC) in CIRCLE(source3_RA,source3_Dec,source3_rad))'"
```

Where source_RA, source_Dec, and source_rad are for the main source; source2_RA, source2_Dec, and source2_rad are for the second source; and source3_RA,source3_Dec, and source3_rad are for the third source. Note, the second two regions in the expression are joined using the `&&!` operator which means `and not`. This will exclude the events inside the two smaller regions while keeping the events in the larger region.

Below we have the RA and Dec for the two neighboring sources and assumed a radius of 7 arcseconds for both. We use a larger 30 arcsecond radius for the region of the main source.

In [ ]:
source_RA  = 213.292 * u.deg # degrees
source_Dec = -65.339 * u.deg # degrees
source_rad = 30.0 * u.arcsec # arcseconds

source2_RA  = 213.3015 * u.deg # degrees
source2_Dec = -65.3375 * u.deg # degrees
source2_rad = 7.0 * u.arcsec # arcseconds

source3_RA  = 213.292 * u.deg # degrees
source3_Dec = -65.346 * u.deg # degrees
source3_rad = 7.0 * u.arcsec # arcseconds

sources = [(source_RA ,source_Dec ,source_rad ),
           (source2_RA,source2_Dec,source2_rad),
           (source3_RA,source3_Dec,source3_rad)]

for inst, file in hi_res_images.items():
    plot_multi_regions(file, sources, source_RA, source_Dec, source_rad)

Now we need to select a background region. Normally you could can select the background using an annulus around your source region. But in this case because there are nearby bright sources you need to select a background region further away.

For extracting a MOS background spectrum, it is recommended to select a region that is on the same CCD, away from any sources. For extracting a pn background spectrum, it is recommended to select a region that doesn't include columns that pass through the source to avoid out-of-time events (this is particularly important for bright sources, and less so for faint ones). The region should also have the same distance to the readout node as the source region, i.e., they have the same RAWY values. For each CCD, RAWY's origin is at the chip edge that corresponds to the outside of the array. So, the source and background regions should have about the same distance to the outer edge of whichever CCD they are on. If that isn't possible, choose a region that is on the same CCD as the source; and if that isn't possible, choose a region in the same quadrant.

With this in mind we will select a background region nearby, but because the psf of the source is so large we have to select a region sufficiently far away to prevent overlap.

In [ ]:
bkg_RA  = 213.28 * u.deg  # degrees
bkg_Dec = -65.453 * u.deg # degrees
bkg_rad = 30.0 * u.arcsec # arcseconds

for inst, file in hi_res_images.items():
    plot_region(file, bkg_RA, bkg_Dec, bkg_rad, vmax=100)

Now we can extract the source and background spectra.

In [ ]:
for inst in clean_event_lists.keys():
    if inst == 'EPN':
        specchannelmax = 20479
    elif 'EMOS' in inst:
        specchannelmax = 11999

    # 11 arcsecond source region
    # Uncomment the following lines to use the smaller source region
    #source_rad = 11.0 * u.arcsec # arcseconds
    #source_region = "CIRCLE({0},{1},{2})".format(source_RA.value,source_Dec.value,source_rad.to(u.deg).value)
    #expression = "'((RA,DEC) in {0})'".format(source_region)

    # 30 arcsecond source region, minus two nearby sources
    # Uncomment the following lines to use the larger source region
    source_rad = 30.0 * u.arcsec # arcseconds
    main_source = "((RA,DEC) in CIRCLE({0},{1},{2}))".format(source_RA.value,source_Dec.value,source_rad.to(u.deg).value)
    scnd_source = "((RA,DEC) in CIRCLE({0},{1},{2}))".format(source2_RA.value,source2_Dec.value,source2_rad.to(u.deg).value)
    thrd_source = "((RA,DEC) in CIRCLE({0},{1},{2}))".format(source3_RA.value,source3_Dec.value,source3_rad.to(u.deg).value)
    expression = "'{0} &&! {1} &&! {2}'".format(main_source,scnd_source,thrd_source)

    inargs = {}
    inargs = {'table'           : clean_event_lists[inst],
              'energycolumn'    : 'PI',
              'withfilteredset' : 'yes',
              'filteredset'     : source_event_list[inst],
              'keepfilteroutput': 'yes',
              'filtertype'      : 'expression',
              'expression'      : expression,
              'withspectrumset' : 'yes',
              'spectrumset'     : source_spectra_file[inst],
              'spectralbinsize' : '5',
              'withspecranges'  : 'yes',
              'specchannelmin'  : '0',
              'specchannelmax'  : specchannelmax}
    
    MyTask('evselect', inargs).run()
    
    bkg_region = "CIRCLE({0},{1},{2})".format(bkg_RA.value,bkg_Dec.value,bkg_rad.to(u.deg).value)

    inargs = {}
    inargs = {'table'           : clean_event_lists[inst],
              'energycolumn'    : 'PI',
              'withfilteredset' : 'yes',
              'filteredset'     : bkg_event_list[inst],
              'keepfilteroutput': 'yes',
              'filtertype'      : 'expression',
              'expression'      : "'((RA,DEC) in {0})'".format(bkg_region),
              'withspectrumset' : 'yes',
              'spectrumset'     : bkg_spectra_file[inst],
              'spectralbinsize' : '5',
              'withspecranges'  : 'yes',
              'specchannelmin'  : '0',
              'specchannelmax'  : specchannelmax}
    
    MyTask('evselect', inargs).run()

We can take a look at the resulting source. The only thing in the image should be the source.

In [ ]:
source_image = {}
for inst, file in source_event_list.items():
    source_image[inst] = f'{inst}_source_image.fits'
    make_hires_image(file,out_image=source_image[inst],output=False)
    plot_zoom_in(source_image[inst], vmax = 1000.0)

## 4. Check for Pile Up

Before we go any futher we have to do one final check for the quality of our data. Because this is a bright source we will have to check for pile up.

In [ ]:
for inst in source_event_list.keys():
    inargs = {'set'               : source_event_list[inst],
              'plotfile'          : epatplot[inst],
              'useplotfile'       : 'yes',
              'withbackgroundset' : 'yes',
              'backgroundset'     : bkg_event_list[inst]}
    
    MyTask('epatplot', inargs).run()

When that is done you will have three plots saved as pdfs in your current work directory for the Obs ID we are working with. If you don't remember where your current work directory is you can run the cell below.

In [ ]:
print(my_obs.work_dir)

The resulting plots should look something like this:

![Circinus Galaxy EMOS1 Pileup Plot](./_files/EMOS1_epatplot_combine_spectra.png)

![Circinus Galaxy EMOS2 Pileup Plot](./_files/EMOS2_epatplot_combine_spectra.png)

![Circinus Galaxy EPN Pileup Plot](./_files/EPN_epatplot_combine_spectra.png)

The results for the MOS cameras look alright, but the pn shows a slight amount of pile up for the events with a double event pattern (note the values for double events s: 1.046 +/- 0.013, which means the deviation from the model is slightly outside the uncertainty) and this may skew the resulting spectrum if we use the double events for our analysis.

## 5. RMF, ARF and grouping the Spectra

We can now generate the ARF and RMF for all three cameras. Then we can group the spectra.

<div class="alert alert-block alert-info">
    <b>Note:</b> The performance (speed) of <tt>rmfgen</tt> is directly proportional to the number of bins in the energy grid. A larger number of bins, which means smaller bin sizes, will greatly increase the time needed to run <tt>rmfgen</tt>. The trade off is that the scientific accuracy of the results improves (to the limit of the energy resolution of the instrument). In the cell below we set the number of bins such that the energy resolution is 0.01 keV (10 eV). Alternatively, you can uncomment the second <tt>NBINS</tt> lines to increase the number of bins by a factor of 10, giving an energy resolution of 0.001 keV (1 eV). This will make <tt>rmfgen</tt> take 10x longer to run (~50 minutes compared to ~5 minutes).
</div>

In [ ]:
for inst in source_spectra_file.keys():
    if inst == 'EPN':
        MAXENERGY = 15
        NBINS     = 1490
        #NBINS     = 14900
    elif 'EMOS' in inst:
        MAXENERGY = 12
        NBINS     = 1190
        #NBINS     = 11900

    inargs = {}
    inargs = {'rmfset'         : rmf_file[inst],
              'spectrumset'    : source_spectra_file[inst],
              'withenergybins' : 'yes',
              'energymin'      : 0.1,
              'energymax'      : MAXENERGY,
              'nenergybins'    : NBINS}
    
    MyTask('rmfgen', inargs).run()
    
    inargs = {}
    inargs = {'arfset'         : arf_file[inst],
              'spectrumset'    : source_spectra_file[inst],
              'withrmfset'     : 'yes',
              'rmfset'         : rmf_file[inst],
              'withbadpixcorr' : 'yes',
              'badpixlocation' : clean_event_lists[inst],
              'setbackscale'   : 'yes'}
    
    MyTask('arfgen', inargs).run()

In [ ]:
for inst in source_spectra_file.keys():
    inargs = {'spectrumset' : source_spectra_file[inst],
              'groupedset'  : grouped_spectra[inst],
              'arfset'      : arf_file[inst],
              'rmfset'      : rmf_file[inst],
              'backgndset'  : bkg_spectra_file[inst],
              'mincounts'   : 20}
    
    MyTask('specgroup', inargs).run()

Now we will combine the three EPIC spectra into a single sprectrum. The equivalent SAS command would be:
```
epicspeccombine pha="EMOS1_pi.fits EMOS2_pi.fits EPN_pi.fits"\
   bkg="EMOS1_bkg_pi.fits EMOS2_bkg_pi.fits EPN_bkg_pi.fits" \
   rmf="EMOS1_rmf.fits EMOS2_rmf.fits EPN_rmf.fits" \
   arf="EMOS1_arf.fits EMOS2_arf.fits EPN_arf.fits" \
   filepha="src_spectrum_grp.ds" \
   filebkg="bkg_spectrum_grp.ds" \
   filersp="response_grp.rmf" \
   allowHEdiff=yes
```

In [ ]:
pha = []
bkg = []
rmf = []
arf = []

for inst in grouped_spectra.keys():
    pha.append(source_spectra_file[inst])
    bkg.append(bkg_spectra_file[inst])
    rmf.append(rmf_file[inst])
    arf.append(arf_file[inst])

pha = " ".join(pha)
bkg = " ".join(bkg)
rmf = " ".join(rmf)
arf = " ".join(arf)

In [ ]:
inargs = {'pha' : pha,
          'bkg' : bkg,
          'rmf' : rmf,
          'arf' : arf,
          'filepha' : filepha,
          'filebkg' : filebkg,
          'filersp' : filersp,
          'allowHEdiff' : True}

MyTask('epicspeccombine', inargs).run()

The spectra for our source are now grouped and ready for analysis in Part 2.